# C1 关系判定编码器：Colab T4 训练

1. 运行时 → 更改运行时类型 → T4 GPU。
2. 运行第一格，上传本机打包的 `c1-bundle.zip`（含 `relation.jsonl` 与 `train_c1.py`，由 `python -m experiments.recognition.relation.bundle` 生成）。
3. 依次运行。主模型约 10 分钟；两个 E2 对照（线性适配、全量微调）可选。
4. 最后一格把 `c1-out.zip` 下载回本机，解压到 `runs/local/recognizers/c1-xlmr/`，再在本机 `.venv-encoder` 下跑 `predict` 与登记（见 handover 文档）。

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import files
uploaded = files.upload()  # choose c1-bundle.zip
!unzip -o c1-bundle.zip -d c1 && ls -la c1

In [ ]:
!pip -q install "transformers==4.57.1" "safetensors==0.6.2"
import torch, transformers; print(torch.__version__, transformers.__version__, torch.cuda.is_available())

In [ ]:
# Main model: freeze the bottom 8 layers, 3 epochs, dev-joint early stopping.
!cd c1 && python train_c1.py --data relation.jsonl --base FacebookAI/xlm-roberta-base --out ../c1-out --freeze-layers 8 --epochs 3 --batch 16 --lr 2e-5

In [ ]:
# E2 (optional): linear adapter (all encoder layers frozen) and full fine-tuning.
!cd c1 && python train_c1.py --data relation.jsonl --out ../c1-linear --freeze-layers 12 --epochs 3 --head-lr 3e-3
!cd c1 && python train_c1.py --data relation.jsonl --out ../c1-full --freeze-layers 0 --epochs 3 --lr 1e-5

In [ ]:
import json
for name in ['c1-out', 'c1-linear', 'c1-full']:
    try:
        m = json.load(open(f'{name}/metrics.json'))
        print(name, {lang: round(m['test'][lang]['macro_f1'], 3) for lang in ('zh', 'en', 'all')}, 'joint', round(m['test']['all']['joint'], 3), 'T', m['temperature'])
    except FileNotFoundError:
        pass
!zip -qr c1-out.zip c1-out
!for d in c1-linear c1-full; do [ -d $d ] && cp $d/metrics.json $d-metrics.json; done; zip -q c1-e2-metrics.zip c1-*-metrics.json 2>/dev/null; ls -la *.zip
files.download('c1-out.zip')

In [ ]:
files.download('c1-e2-metrics.zip')